In [1]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=cm.get_cmap("Dark2").colors)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

# Put the repo's `src/` on sys.path so `quantproteomicssimbox` imports even without an editable
# install. This notebook lives in notebooks/, so the repo's `src/` is at "../src".
src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from quantproteomicssimbox.methods import intensity_method, stoichiometry_method, QUANT_METHODS
from quantproteomicssimbox.sweep import run_sweep, records_to_rows

C:\Users\student\AppData\Local\Temp\ipykernel_24392\4030008377.py:8: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  plt.rcParams['axes.prop_cycle'] = plt.cycler(color=cm.get_cmap("Dark2").colors)


## Experiment Setup

In [7]:
# Sets the experiment parameters. 
# Constant values are passed to the sweep's base/fixed parameters.
# *List* parameters are passed to the sweep's grid parameters.
gridable_params = dict(
    # Experiment truth parameters
    n_proteins = 5,
    protein_length = 200,
    abundance = 250,
    n_groups = 2, # aka conditions
    n_subjects = 15, # aka bioreps
    repeat_units = 0, # peptide repeats in the sequence at different start loci
    digestion = "per_subject", # 'per_subject' or 'per_copy'
    miscleavage_model = ["global", "bernoulli"], # 'global' or 'bernoulli'
    miscleavage_rate = 0.25, 

    # Observation model parameters
    var_subject = [0, 1, 9],
    var_site = [0, 1, 9],
    var_species = [0, 1, 9],
    position_aware = False, # if False, the repeat_units will be aggregated despite representing different spans.
    detection_limit = 0,
    missingness = [0, 0.25, 0.5]
)

# Sets the grid of rollup/quantification metrics. 
# Default imports a set of 7 common quant methods. See `quantproteomicssimbox.methods.QUANT_METHODS`. 
# Additional methods can be created with the imported 
# `intensity_method()` and `stoichiometry_methods()` functions.
quant_methods = QUANT_METHODS

# Group/condition analysis filter. 
# Only peptides in at least that many samples/bioreps of any group/condition are kept for analysis. 
# See `quantproteomicssimbox.rollups.intensity.roll_up()``
min_per_group = 1

# Pseudo-randomness seed
seed = 42

# Number of sweep replicates
n_replicates = 5

# Set up sweep's `base` and `grid` arguments
base = {}
grid = {}
for k,v in gridable_params.items():
    if isinstance(v, list):
        grid[k] = v
    else: 
        base[k] = v
n_experiments = np.prod([len(v) for _,v in grid.items()]) * len(quant_methods)
print("Number of Grid Points:", n_experiments)
print("Total Number of Experiments (including replicates):", n_replicates * n_experiments)

Number of Grid Points: 1134
Total Number of Experiments (including replicates): 5670


## Experiment Run

In [3]:
results = run_sweep(grid=grid, 
                    methods=quant_methods, 
                    n_replicates = n_replicates,
                    base=base, 
                    min_per_group=min_per_group,
                    seed = seed)

## Experiment Analysis

In [4]:
results_for_pd = records_to_rows(results)
df = pd.DataFrame(results_for_pd)
df

,miscleavage_model,var_subject,var_site,var_species,missingness,method,rmse_mean,rmse_se,n
0,global,0,0,0,0.0,int_mean,0.829939,0.139906,5
1,global,0,0,0,0.0,int_median,0.911060,0.144110,5
2,global,0,0,0,0.0,int_sum,2.573305,0.573441,5
3,global,0,0,0,0.0,stoich_fraction,0.004248,0.004248,5
4,global,0,0,0,0.0,stoich_logit,0.049494,0.049494,5
...,...,...,...,...,...,...,...,...,...
1129,bernoulli,9,9,9,0.5,int_sum,2.782564,0.519014,5
1130,bernoulli,9,9,9,0.5,stoich_fraction,1.440461,0.165786,5
1131,bernoulli,9,9,9,0.5,stoich_logit,7.444099,0.540629,5
1132,bernoulli,9,9,9,0.5,stoich_pep_mean,1.440461,0.165786,5


In [12]:
df.to_csv("../results/stoichiometry_sweep.csv")